# 08 · BETO: embeddings contextuales

**Etapa 6 del pipeline del Proyecto 2:** representaciones contextuales con **BETO** (`dccuchile/bert-base-spanish-wwm-cased`)


In [1]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

# Raíz del proyecto: el notebook se ejecuta desde la carpeta notebooks/
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ / "src"))

import embeddings_beto as eb
import embeddings_w2v as ew

NUMERO_EQUIPO = 1          # <- cambiar por el número de equipo (igual que en el notebook 07)
SEMILLA = NUMERO_EQUIPO

pd.set_option("display.max_colwidth", 220)
df = ew.cargar_corpus()
print(f"{len(df)} reseñas en el corpus")

16991 reseñas en el corpus


## 0. Carga de BETO desde HuggingFace

BETO es BERT entrenado con un corpus grande en español. Se carga el tokenizador y el modelo base (sin capa de clasificación), en modo evaluación. La primera vez se descarga (~440 MB).

In [2]:
beto = eb.cargar_beto()
config = beto["modelo"].config

pd.Series({
    "modelo": eb.MODELO_BETO,
    "dispositivo": beto["dispositivo"],
    "parámetros": f"{sum(p.numel() for p in beto['modelo'].parameters()):,}",
    "dimensión del embedding": config.hidden_size,
    "capas (Transformer)": config.num_hidden_layers,
    "cabezas de atención": config.num_attention_heads,
    "vocabulario (subpalabras)": f"{beto['tok'].vocab_size:,}",
    "máximo de tokens": config.max_position_embeddings,
})

C:\Users\ale23\Desktop\CUC\REPOS-CUC\ProyectoExploracion\MineriaDatos\src\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 14059.76it/s]
[transformers] BertModel LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
pooler.dense.bi

modelo                       dccuchile/bert-base-spanish-wwm-cased
dispositivo                                                    cpu
parámetros                                             109,850,880
dimensión del embedding                                        768
capas (Transformer)                                             12
cabezas de atención                                             12
vocabulario (subpalabras)                                   31,002
máximo de tokens                                               512
dtype: object

In [3]:
# Cómo divide BETO el texto en subpalabras (WordPiece)
ejemplo = "La cinematografía de esta película es sobrecogedora."
print(beto["tok"].tokenize(ejemplo))

['La', 'cinemato', '##grafía', 'de', 'esta', 'película', 'es', 'sobre', '##co', '##gedor', '##a', '.']


### 1 Polisemia en el corpus

Se toman fragmentos reales de las reseñas que contienen cada palabra, se calcula su embedding con BETO y se agrupan con **K-Means (k = 2)** sin decirle al algoritmo cuáles son los sentidos. Si los grupos corresponden a sentidos distintos, BETO los distingue también en texto real.

In [4]:
def resaltar(fragmento, palabra, ancho=12):
    # Fragmento corto alrededor de la palabra, con la palabra en MAYÚSCULAS
    palabras = fragmento.split()
    pos = next((i for i, p in enumerate(palabras) if palabra in p.lower()), 0)
    trozo = palabras[max(0, pos - ancho):pos + ancho + 1]
    return " ".join(p.upper() if palabra in p.lower() else p for p in trozo)

usos_corpus = {}
for palabra in ["papel", "estrella", "crítica", "corto", "plano", "final"]:
    contextos = eb.contextos_del_corpus(df, palabra, n=40, semilla=SEMILLA)
    vectores, validos = eb.embeddings_palabra(contextos["fragmento"].tolist(), palabra, beto)
    contextos = contextos.iloc[validos].copy()
    contextos["grupo"] = KMeans(n_clusters=2, random_state=SEMILLA, n_init=10).fit_predict(vectores)
    usos_corpus[palabra] = (contextos, vectores)

    print(f"\n=== {palabra} ({len(contextos)} usos) ===")
    for grupo, g in contextos.groupby("grupo"):
        print(f"  Grupo {grupo} ({len(g)} usos):")
        for fragmento in g["fragmento"].head(3):
            print(f"    · …{resaltar(fragmento, palabra)}…")


=== papel (40 usos) ===
  Grupo 0 (37 usos):
    · …Norman Bates. Nadie podría haber elegido a un hombre mejor para el PAPEL. Él es tan lindo :) Es una película maravillosa ... y hermosamente…
    · …era la imaginación de dar vida a un oso y cambiar el PAPEL de él de ser sólo un juguete infantil. Ted es graciosamente divertido…
    · …personaje de Mina estaba completamente... ¡Completamente destruido y cambiado!!! De hecho, su PAPEL fue cambiado por el de Drácula. Parece estar bien, y puedes empatizar…
  Grupo 1 (3 usos):
    · …mayor parte de ella bastante bien. En una de las versiones en PAPEL del material fuente THE MIDWICH CUCKOOS fue una mini guía de estudio,…
    · …película es que WALL-E no separa las montañas de basura en metales, PAPEL, plásticos, etc, para reciclar, sino que sólo lo comprime en cubos convenientes…
    · …antes de que se vuelvan tan interesantes para el público como el PAPEL pintado o la música de ascensor. Tal vez este es una indicación…

=== estrella (40 us

---
## 2. Búsqueda semántica de reseñas

Se calcula el embedding de **todas las reseñas del corpus** con el vector **[CLS]** de BETO, como en la sección 3.3 del laboratorio: el primer token, pensado para representar la oración completa (es el campo `beto_cls` del esquema de MongoDB).

Una consulta en lenguaje natural se representa igual y se buscan las reseñas con mayor similitud coseno.

### Configuración

- `MAX_TOKENS = 512`: el máximo que admite BETO; las preguntas frecuentes del enunciado indican truncar a 512 tokens. Solo se recorta la parte final de las reseñas más largas.
- `LOTE`: reseñas por lote (el enunciado sugiere procesar en lotes pequeños).
- **Tiempo:** en CPU son aproximadamente **2 horas** para las ~17 000 reseñas; en Google Colab con GPU, unos 10-15 minutos.
- El avance se guarda en `data/results/embeddings_beto_parcial.npz`: si se interrumpe (o se cierra el notebook), al volver a ejecutar la celda continúa donde quedó.

In [5]:
MAX_TOKENS = 512            # máximo de BETO (preguntas frecuentes del enunciado)
LOTE = 16

resenas = df.reset_index(drop=True)
print(f"Reseñas a procesar: {len(resenas)} · tiempo estimado en CPU: ~{len(resenas) / 2.5 / 3600:.1f} horas")

Reseñas a procesar: 16991 · tiempo estimado en CPU: ~1.9 horas


In [7]:
X_cls = eb.embeddings_resenas(resenas["texto_es"], beto, lote=LOTE, max_tokens=MAX_TOKENS,
                              ruta_parcial=eb.RUTA_PARCIAL)
print(f"Embeddings [CLS]: {X_cls.shape}")

Reanudando: 16991 de 16991 reseñas ya procesadas


0lote [00:00, ?lote/s]

Embeddings [CLS]: (16991, 768)


In [8]:
def buscar_resenas(consulta, n=5):
    # Reseñas del corpus más parecidas a la consulta (vector [CLS])
    vector = eb.embedding_texto(consulta, beto, max_tokens=MAX_TOKENS)
    indices, similitudes = eb.buscar(vector, X_cls, n=n)
    resultado = resenas.iloc[indices][["id", "titulo_obra", "genero_es", "polaridad", "origen", "texto_es"]].copy()
    resultado.insert(0, "similitud", similitudes.round(3))
    resultado["texto_es"] = resultado["texto_es"].str[:250] + "…"
    return resultado.reset_index(drop=True)

consultas = [
    "una película lenta pero con una fotografía preciosa",
    "efectos especiales impresionantes pero una historia floja",
    "me hizo llorar de emoción, una historia muy conmovedora",
    "una película de terror que da mucho miedo",
    "ideal para ver con niños, muy divertida y tierna",
    "la peor película que he visto, aburrida y sin sentido",
]
for consulta in consultas:
    print(f"\n### Consulta: «{consulta}»")
    display(buscar_resenas(consulta))


### Consulta: «una película lenta pero con una fotografía preciosa»


,similitud,id,titulo_obra,genero_es,polaridad,origen,texto_es
0,0.860,16532,One Day (Siempre el mismo día),Drama,positiva,nativo,Quiero decir que esta película tiene todo lo que uno quiere ver da una gran enseñanza la recomiendo mucho y más si quiere verla con tu pareja me llego mucho esa película si la gente la mirara bien la mira de un punto...
1,0.858,15765,Las Aventuras de Tadeo Jones,Animación,positiva,nativo,me gusto mucho y a mi niña también me gusto el argumento por ser un héroe de origen humilde un simple albañil y tiene buen fondo y encima es española…
2,0.853,15078,Los juegos del hambre: En llamas,Acción,negativa,nativo,"me ha parecido una película muy lenta! la primera hora y media ha sido eterna por favor! cuando más o menos empezaba a ponerse interesante, sorpresa!!! se acaba y a esperar la tercera parte. hubiese preferido verla e..."
3,0.851,15229,Deadpool,Acción,positiva,nativo,"Por fin fox hace que una película marvell esté a la altura, película muy bien hecha gracia a que se les a dejado libertad para hacerla....y el actor que decir....lo borda…"
4,0.851,16339,Cincuenta sombras más oscuras,Drama,positiva,nativo,me encanto excelente y las escenas de sexo estuvieron mas reales. estuvo mejor que la primera. la espera de dos años valió la pena fui dos veces al cine a verla jejje. me encantan los actores tienen una química que p...



### Consulta: «efectos especiales impresionantes pero una historia floja»


,similitud,id,titulo_obra,genero_es,polaridad,origen,texto_es
0,0.901,15719,El Rey León,Animación,intermedia,nativo,"A pesar del maravilloso CGI y las correctas perfomances vocales, la película carece de la magia, el alma y el corazón de la original Este live-action solo se queda en un buen intento…"
1,0.899,15443,Enredados (Rapunzel),Animación,intermedia,nativo,Una pelicula mas de Disney pero maravillosa al fin y a el cabo para verla con la familia para verla todos sentado muy buenos toques cómicos sin olvidarse de la trama principal es maravillosa…
2,0.883,15904,Los descendientes,Comedia,negativa,nativo,Desendientes una película para llorar pero de odio mi primo de 4 piensa mejores cosas que desendientes la trama no tiene ni pies ni cabeza las canciones cuentan toda la película…
3,0.881,15363,Piratas del Caribe: La venganza de Salazar,Acción,positiva,nativo,Me a gustado mucho como las anteriores si eres fan de la saga te gustara el papel del capitán Salazar le da un nuevo frescor a la peli increíble a mi modo de ver gran peli…
4,0.879,15324,Dragon Ball Super: Broly,Acción,positiva,nativo,excelente como siempre draagon ball super broly es una buena película la animación es de primera la mejor del 2018 y de todas las peliculas de dragon ball z las peleas fueron lo mejor…



### Consulta: «me hizo llorar de emoción, una historia muy conmovedora»


,similitud,id,titulo_obra,genero_es,polaridad,origen,texto_es
0,0.867,16046,Focus,Comedia,intermedia,nativo,"Me gusto, lo unico que el final no es tan facil de entender, habría que verla dos veces para entenderlo, es un giro inesperado, me hizo acordar al final de contratiempo…"
1,0.862,15765,Las Aventuras de Tadeo Jones,Animación,positiva,nativo,me gusto mucho y a mi niña también me gusto el argumento por ser un héroe de origen humilde un simple albañil y tiene buen fondo y encima es española…
2,0.858,16246,Nymphomaniac. Volumen 1,Drama,intermedia,nativo,"Me ha gustado y la verdad es que no se deja censurar por nada esta película, la historia de Joe es bastante interesante y muy intrigante y Seligman me ha agradado mucho.…"
3,0.851,15607,Rio,Animación,positiva,nativo,"Esta película me ha enamorado, me rio de los que dicen que es para bebés, a mi me encantó, y la volvería a ver mil veces, y tengo 18 años.…"
4,0.849,15512,Los Increíbles 2,Animación,positiva,nativo,Me encantó la avía esperado desde niño y ojalá no tarde tanto como la segunda su tercera entrega sin duda voy y la veo en cualquier cine de mi ciudad …


... (2 outputs left out)


### Consulta: «ideal para ver con niños, muy divertida y tierna»


,similitud,id,titulo_obra,genero_es,polaridad,origen,texto_es
0,0.831,16352,Wonder,Drama,positiva,nativo,"Apreciar las almas, nunca los rostros. Aborda la problemática infantil de la deformidad facial, la inserción social del protagonista y el apoyo familiar para un desarrollo pleno. Súper cálida, emotiva y sobre todo in..."
1,0.823,15390,Venom,Acción,intermedia,nativo,"accion, aventuras, y buenos protagonistas. hasta los malos que en un principio restaban a la nota final, creo que aportan el punto justo propio de la generación que crecimos en los 80. Muy recomendable.…"
2,0.821,15863,Intocable,Comedia,positiva,nativo,"impresionante, es imposible que haya una persona en este mundo al que no le encante esta pelicula! no apta para personas de lagrima facil, muy buena divertida y bonita sobre todo…"
3,0.814,15305,Las aventuras de Tintín: El secreto del unicornio,Acción,positiva,nativo,"Película muy guapa,aventuras y risas....lo que no se quien se copio de quien si spielberg del equipo que creo uncharted 3 o al revés,en fin recomendable para todas las edades…"
4,0.805,15603,Rio,Animación,positiva,nativo,"Es muy divertida, ideal para ir a verla con niños ya sean pequeños o mas mayorcitos, tiene mucha acción y colores, eso hace que los mas pequeñines aguanten viendola, y tiene algunos puntillos que nos sacara una sonri..."



### Consulta: «la peor película que he visto, aburrida y sin sentido»


,similitud,id,titulo_obra,genero_es,polaridad,origen,texto_es
0,0.888,16333,Tengo ganas de ti,Drama,positiva,nativo,Me encanto es una de las mejores películas de la vida es algo muy bonito que nos da a creer más en el amor y en tu pareja que real mente amas sos la mejor película de amor de todos los tiempos…
1,0.887,15929,Vacaciones,Comedia,negativa,nativo,"Tiene partes divertidas, pero me parece demasiado grosera, osea como es posible que un niño de tan poca edad interprete ese tipo de dialgo y escenas, no es una gran película la verdad…"
2,0.887,15443,Enredados (Rapunzel),Animación,intermedia,nativo,Una pelicula mas de Disney pero maravillosa al fin y a el cabo para verla con la familia para verla todos sentado muy buenos toques cómicos sin olvidarse de la trama principal es maravillosa…
3,0.886,15344,Colombiana,Acción,positiva,nativo,está esta película se trata sobre una chica que fue secuestrada y mataron a sus padres es una película de mucho suspenso y acción si le gustan las películas de venganza Esta es una buena película…
4,0.881,15043,Escuadrón suicida,Acción,negativa,nativo,"Voy mucho al cine y es de las peores películas que he visto. El argumento no tiene sentido, los efectos especiales son buenos pero no para tirar cohetes pero igualmente sin sentido…"


---
## 3. Masked Language Model

BETO fue preentrenado prediciendo palabras tapadas con `[MASK]`. Se analiza cómo completa frases típicas de reseñas y si sus predicciones coinciden con el vocabulario de reseñas positivas o negativas del corpus.

Para clasificar las predicciones se construye un **léxico de polaridad del propio corpus**: para cada palabra, el logaritmo del cociente entre su frecuencia relativa en reseñas positivas y en negativas (> 0,5 = positiva, < −0,5 = negativa, entre ambos = neutra; *sin datos* si la palabra aparece menos de 20 veces).

In [9]:
fill_mask = eb.cargar_fill_mask()
lexico = eb.lexico_polaridad(df["texto_es"], df["polaridad"])
print(f"Léxico de polaridad: {len(lexico):,} palabras")
pd.Series({p: round(lexico.get(p, np.nan), 2) for p in
           ["excelente", "obra", "genial", "buena", "aceptable", "aburrida", "mala", "basura", "pérdida"]},
          name="log(frec. positivas / frec. negativas)")

Loading weights: 100%|██████████| 204/204 [00:00<00:00, 15777.36it/s]


Léxico de polaridad: 7,687 palabras


excelente    1.43
obra         0.82
genial       0.82
buena        0.17
aceptable   -1.04
aburrida    -1.98
mala        -1.50
basura      -2.24
pérdida     -1.82
Name: log(frec. positivas / frec. negativas), dtype: float64

### 3.1 Frases típicas de reseñas (enunciado)

In [10]:
plantillas = [
    "La película fue [MASK].",
    "La actuación del protagonista fue [MASK].",
    "Los efectos especiales son [MASK].",
    "La película me pareció muy [MASK].",
    "El guion es [MASK].",
    "La banda sonora es [MASK].",
]
predicciones = eb.predecir_mascara(fill_mask, plantillas, lexico, n=5)
predicciones

,oración,predicción,probabilidad,polaridad en el corpus
0,La película fue [MASK].,estrenada,0.1110,sin datos
1,La película fue [MASK].,titulada,0.0859,sin datos
2,La película fue [MASK].,prohibida,0.0766,sin datos
3,La película fue [MASK].,de,0.0390,neutra
4,La película fue [MASK].,distribuida,0.0389,sin datos
5,La actuación del protagonista fue [MASK].,excelente,0.0897,positiva
6,La actuación del protagonista fue [MASK].,buena,0.0534,neutra
7,La actuación del protagonista fue [MASK].,escasa,0.0415,sin datos
8,La actuación del protagonista fue [MASK].,fantástica,0.0371,positiva
9,La actuación del protagonista fue [MASK].,notable,0.0339,positiva


### 3.2 ¿Cambian las predicciones si se antepone el género?

In [11]:
generos_texto = {"Terror": "terror", "Comedia": "comedia", "Acción": "acción",
                 "Animación": "animación", "Drama": "drama"}
filas = []
for genero, nombre in generos_texto.items():
    for plantilla in ["Esta película de {} fue [MASK].", "Esta película de {} es muy [MASK]."]:
        oracion = plantilla.format(nombre)
        p = eb.predecir_mascara(fill_mask, [oracion], lexico, n=5)
        filas.append({"género": genero, "oración": oracion,
                      "predicciones": ", ".join(f"{w} ({s:.2f})" for w, s in zip(p["predicción"], p["probabilidad"])),
                      "positivas": (p["polaridad en el corpus"] == "positiva").sum(),
                      "negativas": (p["polaridad en el corpus"] == "negativa").sum()})
pd.DataFrame(filas).set_index(["género", "oración"])

predicciones  \
género    oración                                                                                                                            
Terror    Esta película de terror fue [MASK].          prohibida (0.34), estrenada (0.11), producida (0.07), lanzada (0.06), creada (0.04)   
          Esta película de terror es muy [MASK].           popular (0.09), buena (0.08), divertida (0.05), rara (0.04), interesante (0.04)   
Comedia   Esta película de comedia fue [MASK].       prohibida (0.25), estrenada (0.15), lanzada (0.06), producida (0.05), titulada (0.03)   
          Esta película de comedia es muy [MASK].       popular (0.18), buena (0.08), exitosa (0.06), divertida (0.04), interesante (0.04)   
Acción    Esta película de acción fue [MASK].        prohibida (0.22), estrenada (0.08), producida (0.07), lanzada (0.05), titulada (0.05)   
          Esta película de acción es muy [MASK].     popular (0.08), buena (0.06), interesante (0.04), divertida (0.04), importante (0.02)   
Animación Esta película de animación fue [MASK].     prohibida (0.15), lanzada (0.08), producida (0.07), estrenada (0.06), titulada (0.04)   
          Esta película de animación es muy [MASK].     popular (0.27), buena (0.06), interesante (0.04), exitosa (0.04), divertida (0.03)   
Drama     Esta película de drama fue [MASK].         estrenada (0.12), producida (0.10), lanzada (0.10), prohibida (0.08), titulada (0.05)   
          Esta película de drama es muy [MASK].              popular (0.21), buena (0.07), exitosa (0.05), interesante (0.03), rara (0.03)   

                                                     positivas  negativas  
género    oración                                                          
Terror    Esta película de terror fue [MASK].                0          0  
          Esta película de terror es muy [MASK].             1          0  
Comedia   Esta película de comedia fue [MASK].               0          0  
          Esta película de comedia es muy [MASK].            1          0  
Acción    Esta película de acción fue [MASK].                0          0  
          Esta película de acción es muy [MASK].             1          0  
Animación Esta película de animación fue [MASK].             0          0  
          Esta película de animación es muy [MASK].          1          0  
Drama     Esta película de drama fue [MASK].                 0          0  
          Esta película de drama es muy [MASK].              0          0

---
## 5. Guardado

- `data/results/embeddings_beto.npz`: vector **[CLS]** de cada reseña (`ids`, `vectores`). La migración a MongoDB lo guarda en `embeddings.beto_cls` (volver a ejecutar el notebook 05) y lo usa el notebook 09.

In [ ]:
eb.guardar_embeddings(resenas["id"], X_cls)
print("Guardado:", eb.RUTA_EMBEDDINGS_BETO.relative_to(RAIZ), X_cls.shape)